# Module 2 - Territory Underperformance

Objective: compare a city's monthly sales with its demographic potential.

Population is only a proxy. Quadrants help prioritize cities, but do not prove commercial undercoverage.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.territory_analysis import (
    aggregate_regions,
    build_city_table,
    classify_territories,
    top_opportunities,
)

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'transactions_normalized.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'generated'

## 1. Load and Check Coverage

Cities without population or a canonical region are excluded from the territorial model but counted in the coverage check.

In [ ]:
transactions = pd.read_csv(INPUT_PATH, parse_dates=['Date'])
print(f'Lignes : {len(transactions):,}')
print(transactions.groupby('Country').size())
print(f'Population manquante : {transactions["Population"].isna().sum():,}')

## 2. Aggregate at City Level

Performance is calculated as `Sales / ActiveMonths` to compare the German and Polish observation windows.

In [ ]:
city = build_city_table(transactions)
city[['City', 'Country', 'Population', 'Sales', 'MonthlySales', 'RegionCanonical']].head()

## 3. Population-Performance Model

Fit a linear regression on logarithms. The residual shows whether a city sells more or less than its population would suggest.

In [ ]:
classified, model, population_median = classify_territories(city)
print(f'Villes utilisées : {len(classified):,}')
print(f'Médiane de population : {population_median:,.0f}')
predicted_log_sales = model.predict(np.log(classified[['Population']].to_numpy()))
actual_log_sales = np.log(classified['MonthlySales'])
r_squared = 1 - ((actual_log_sales - predicted_log_sales) ** 2).sum() / ((actual_log_sales - actual_log_sales.mean()) ** 2).sum()
print(f'R² : {r_squared:.3f}')
classified['Quadrant'].value_counts()

## 4. Prepare Power BI Tables

The three outputs are city, regional and opportunity tables. They are regenerated on every run.

In [ ]:
regions = aggregate_regions(classified)
opportunities = top_opportunities(classified)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
classified.to_csv(OUTPUT_DIR / 'territories_by_city.csv', index=False)
regions.to_csv(OUTPUT_DIR / 'territories_by_region.csv', index=False)
opportunities.to_csv(OUTPUT_DIR / 'territory_opportunities.csv', index=False)
opportunities.head(10)